# RAG (Retrieval-Augmented Generation) con LangChain + OpenAI + Pinecone

Este notebook implementa paso a paso un sistema **RAG** completo:

1. **Indexación**: cargar documentos → dividir en chunks → generar embeddings → almacenar en Pinecone.
2. **Recuperación y generación**: dado un query del usuario → recuperar chunks relevantes → generar respuesta con GPT.

**Stack tecnológico**
| Componente | Tecnología |
|---|---|
| LLM | OpenAI GPT-4o-mini |
| Embeddings | OpenAI text-embedding-3-small (1536 dim) |
| Vector Store | Pinecone Serverless |
| Orquestación | LangChain (LCEL) |
| Fuente de datos | Blog post "LLM Powered Autonomous Agents" – Lilian Weng |

> **Referencia**: [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/) · [Pinecone Integration](https://python.langchain.com/docs/integrations/vectorstores/pinecone)

## 1. Instalación de dependencias

In [ ]:
%pip install -qU \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-pinecone \
    langchain-text-splitters \
    langchain-core \
    pinecone \
    sentence-transformers \
    beautifulsoup4 \
    python-dotenv

print("✅ Dependencias instaladas.")

## 2. Configuración de claves API y variables de entorno

Carga las claves desde un archivo `.env` (recomendado) o ingrésalas directamente.

In [ ]:
import os
from dotenv import load_dotenv

# Suprimir advertencia de USER_AGENT al usar WebBaseLoader
os.environ.setdefault("USER_AGENT", "RAG-LangChain-Demo/1.0")

# Carga variables desde .env si existe en la carpeta raíz del proyecto
load_dotenv(dotenv_path="../.env")

# Si no tienes .env, descomenta y rellena manualmente:
# os.environ["GROQ_API_KEY"]    = "gsk_..."
# os.environ["PINECONE_API_KEY"] = "tu-clave-pinecone"

# Verificar que las claves estén disponibles
assert os.getenv("GROQ_API_KEY"),     "❌ Falta GROQ_API_KEY  → https://console.groq.com/keys"
assert os.getenv("PINECONE_API_KEY"), "❌ Falta PINECONE_API_KEY → https://app.pinecone.io/"
print("✅ Claves API cargadas correctamente.")

## 3. Carga de documentos

Utilizamos `WebBaseLoader` para descargar el contenido del blog post de Lilian Weng sobre agentes autónomos impulsados por LLMs. Este documento servirá como base de conocimiento del RAG.

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://lilianweng.github.io/posts/2023-06-23-agent/"

# Solo conservamos el título, cabeceras y contenido del post
bs4_strainer = bs4.SoupStrainer(
    class_=("post-title", "post-header", "post-content")
)

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    bs_kwargs={"parse_only": bs4_strainer},
)

docs = loader.load()

print(f"Documentos cargados : {len(docs)}")
print(f"Total de caracteres : {sum(len(d.page_content) for d in docs):,}")
print(f"\nPrimeros 500 caracteres:\n{docs[0].page_content[:500]}")

## 4. División de documentos en chunks

El documento completo (~43 000 caracteres) no cabe en el contexto del LLM de forma directa. Lo dividimos en fragmentos más pequeños con solapamiento para preservar el contexto entre chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # máximo de caracteres por chunk
    chunk_overlap=200,  # solapamiento entre chunks consecutivos
    add_start_index=True,
)

all_splits = text_splitter.split_documents(docs)

print(f"Número de chunks generados: {len(all_splits)}")
print(f"\nEjemplo – chunk #0 ({len(all_splits[0].page_content)} chars):")
print(all_splits[0].page_content[:400])

## 5. Generación de embeddings con HuggingFace (gratuito, sin API key)

Usamos el modelo `all-MiniLM-L6-v2` de HuggingFace que corre **localmente** en tu máquina.  
Convierte cada fragmento de texto en un vector de **384 dimensiones**.  
No requiere ninguna clave de API ni costo adicional.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Modelo local gratuito: all-MiniLM-L6-v2 (384 dimensiones)
# Se descarga automáticamente la primera vez (~90 MB)
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print(f"Cargando modelo de embeddings '{EMBEDDING_MODEL}' ...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
)

# Prueba rápida: vectorizar una frase de ejemplo
muestra = "¿Qué es la descomposición de tareas?"
vector_muestra = embeddings.embed_query(muestra)
print(f"Modelo         : {EMBEDDING_MODEL}")
print(f"Dimensión      : {len(vector_muestra)}")
print(f"Primeros 5 val : {[round(v, 4) for v in vector_muestra[:5]]}")

## 6. Almacenamiento de vectores en Pinecone

Creamos (o conectamos a) un índice Pinecone Serverless y subimos todos los chunks ya embebidos.

> **Nota**: la primera vez que ejecutes esta celda puede tardar ~1–2 minutos porque crea el índice y sube todos los vectores.

In [ ]:
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "rag-groq-demo")
DIMENSION  = 384   # dimensión de all-MiniLM-L6-v2

# ── Inicializar cliente Pinecone ────────────────────────────────────────────
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# ── Crear índice si no existe ───────────────────────────────────────────────
if not pc.has_index(INDEX_NAME):
    print(f"Creando índice '{INDEX_NAME}' (dim={DIMENSION}) ...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print("✅ Índice listo.")
else:
    print(f"✅ Índice '{INDEX_NAME}' ya existe.")

# ── Conectar al índice y subir documentos ──────────────────────────────────
index = pc.Index(INDEX_NAME)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)
ids = vector_store.add_documents(documents=all_splits)

print(f"\n✅ {len(ids)} fragmentos almacenados en Pinecone.")
print(f"Ejemplo de IDs: {ids[:3]}")

## 7. Creación del Retriever

Convertimos el vector store en un `Retriever` de LangChain. Cuando recibe una consulta, este objeto:  
1. Convierte la consulta en un vector (usando el mismo modelo de embeddings).  
2. Busca los `k` vectores más cercanos en Pinecone.  
3. Devuelve los documentos correspondientes.

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},  # recuperar los 4 fragmentos más relevantes
)

# Prueba de búsqueda directa antes de construir la cadena completa
consulta_prueba = "¿Qué es la descomposición de tareas?"
docs_recuperados = retriever.invoke(consulta_prueba)

print(f"Consulta de prueba: '{consulta_prueba}'")
print(f"Fragmentos recuperados: {len(docs_recuperados)}\n")
for i, doc in enumerate(docs_recuperados, 1):
    fuente = doc.metadata.get("source", "?")
    inicio = doc.metadata.get("start_index", "?")
    print(f"[{i}] {fuente}  |  inicio={inicio}")
    print(f"    {doc.page_content[:200].replace(chr(10),' ')}...")
    print()

## 8. Construcción de la cadena RAG con LangChain LCEL

La cadena RAG combina:

```
Query → Retriever → Contexto → Prompt → LLM → Respuesta
```

Usamos **LCEL** (LangChain Expression Language) con el operador `|` para encadenar los pasos de forma declarativa.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# ── Modelo de lenguaje (Groq – gratuito) ───────────────────────────────────
# Modelos disponibles: llama-3.3-70b-versatile | llama-3.1-8b-instant | gemma2-9b-it
LLM_MODEL = os.getenv("GROQ_LLM_MODEL", "llama-3.3-70b-versatile")
llm = ChatGroq(model=LLM_MODEL, temperature=0)

# ── Plantilla de prompt ────────────────────────────────────────────────────
PROMPT_SISTEMA = """Eres un asistente experto que responde preguntas sobre \
agentes de IA, planificación, memoria y uso de herramientas, basándote \
exclusivamente en posts de investigación.

Usa ÚNICAMENTE el contexto siguiente para responder. Si el contexto no \
contiene suficiente información, responde: \
"No lo sé con base en el contexto proporcionado."

Contexto:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", PROMPT_SISTEMA),
    ("human", "{question}"),
])

# ── Función auxiliar ───────────────────────────────────────────────────────
def formatear_docs(docs):
    """Concatena los documentos recuperados en un solo string de contexto."""
    return "\n\n".join(doc.page_content for doc in docs)

# ── Cadena LCEL ────────────────────────────────────────────────────────────
# Paso 1: recuperar fragmentos desde Pinecone
# Paso 2: formatear docs → string de contexto
# Paso 3: ejecutar prompt → LLM (Groq) → parser de salida
# Paso 4: devolver respuesta + fuentes citadas

rag_chain = (
    RunnablePassthrough.assign(
        docs=RunnableLambda(lambda x: retriever.invoke(x["question"]))
    )
    | RunnablePassthrough.assign(
        context=lambda x: formatear_docs(x["docs"])
    )
    | RunnablePassthrough.assign(
        answer=(prompt | llm | StrOutputParser())
    )
    | RunnableLambda(
        lambda x: {"answer": x["answer"], "sources": x["docs"]}
    )
)

print(f"✅ Cadena RAG construida  |  LLM: {LLM_MODEL}  |  Embeddings: {EMBEDDING_MODEL}")

## 9. Consultas al sistema RAG

Probamos el pipeline completo con varias preguntas. Para cada consulta el sistema:
1. Recupera los 4 chunks más relevantes de Pinecone.
2. Los inyecta como contexto en el prompt.
3. Genera una respuesta con GPT-4o-mini.
4. Devuelve la respuesta junto con las fuentes citadas.

In [ ]:
def ask_rag(question: str) -> None:
    """Ejecuta la cadena RAG e imprime respuesta + fuentes."""
    print("=" * 65)
    print(f"❓ PREGUNTA: {question}")
    print("=" * 65)

    result = rag_chain.invoke({"question": question})

    print("🤖 RESPUESTA:")
    print(result["answer"])

    sources = result["sources"]
    if sources:
        print(f"\n📚 FUENTES ({len(sources)} chunks recuperados):")
        for i, doc in enumerate(sources, 1):
            src = doc.metadata.get("source", "desconocido")
            offset = doc.metadata.get("start_index", "?")
            snippet = doc.page_content[:120].replace("\n", " ")
            print(f"  [{i}] {src}  (offset {offset})")
            print(f"       \"{snippet}...\"")
    print()


# ── Pregunta 1: Descomposición de tareas ───────────────────────────────────
ask_rag("What is task decomposition and what are its common approaches?")

In [ ]:
# ── Pregunta 2: Tipos de memoria en agentes ────────────────────────────────
ask_rag("What are the different types of memory used in LLM-powered agents?")

In [ ]:
# ── Pregunta 3: Uso de herramientas externas ───────────────────────────────
ask_rag("How do LLM agents use external tools and APIs?")

In [ ]:
# ── Pregunta 4: Pregunta fuera del contexto (prueba de alucinación) ─────────
ask_rag("What is the capital of France?")

## Resumen del pipeline RAG implementado

| Paso | Descripción | Tecnología usada |
|------|-------------|-----------------|
| 1. Carga | Descarga del blog post de Lilian Weng | `WebBaseLoader` + `BeautifulSoup` |
| 2. División | Fragmentación en chunks de 1000 chars con 200 de solapamiento | `RecursiveCharacterTextSplitter` |
| 3. Embedding | Conversión de texto a vectores de 1536 dimensiones | `OpenAIEmbeddings` (`text-embedding-3-small`) |
| 4. Almacenamiento | Upsert de vectores en índice Pinecone Serverless | `PineconeVectorStore` |
| 5. Recuperación | Búsqueda semántica por similitud coseno | `retriever.invoke()` |
| 6. Generación | Respuesta contextualizada con prompt + LLM | `ChatOpenAI` (`gpt-4o-mini`) + LCEL |

### Diagrama de flujo

```
[Usuário] ──query──▶ [Embeddings] ──▶ [Pinecone Search]
                                              │
                                         top-k docs
                                              │
                                         [Prompt]
                                              │
                                          [GPT-4o-mini]
                                              │
                                         [Respuesta]
```

**Referencias**
- [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- [Pinecone Vector Store Integration](https://python.langchain.com/docs/integrations/vectorstores/pinecone)
- [LLM Powered Autonomous Agents – Lilian Weng](https://lilianweng.github.io/posts/2023-06-23-agent/)